# 2. Schema violations

Reproduce the saved schema failure, then optionally run your Responses API call on the reconstructed input. The model call is disabled by default and has not been executed.


In [1]:
import site
from pathlib import Path

venv = Path("/home/hema/research_finder/.venv")
packages = list(venv.glob("lib/python*/site-packages"))
assert packages, "Project environment not found"
site.addsitedir(str(packages[0]))

import openai
import pymupdf
print("Dependencies loaded")

Dependencies loaded


In [2]:
from pathlib import Path
import sys
import json
from pprint import pprint

# Works when opened from the repository, notebooks/, or its parent directory.
root = next(p for base in [Path.cwd(), *Path.cwd().parents]
            for p in [base, base / "research_finder"]
            if (p / "novelty/comparison_workflow.py").exists())
sys.path.insert(0, str(root))
# Check the selected kernel before importing project modules.
import importlib.util
missing = [name for name in ("openai", "pydantic", "pymupdf")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Kernel {sys.executable} is missing: {', '.join(missing)}. "
        f"In VS Code choose Select Kernel > Python Environments > {root / '.venv/bin/python'}, "
        "then restart the kernel and Run All."
    )
from notebooks.example_helpers import load_example, expect_failure
from extraction.research_extract import _parse_json
from novelty.comparison_records import EvidenceReport, EvidenceRecord, EvidenceReview
from novelty.comparison_schemas import Passage
from novelty.comparison_workflow import validate_evidence_report, validate_record, eligible, scope_citation_gaps
from novelty.schemas import TargetSignature


In [3]:
example = load_example(root, "02_schema_violations")
pprint(example["provenance"])

{'authenticity': 'actual saved failed review response; may be an intermediate '
                 'attempt',
 'checkpoint': 'comparison.json',
 'checkpoint_sha256': '9fb5f6b9ffc43d14a7e1b606e9a5cb29e6f3f237a2ef2285c56b1338bea35c67',
 'direction_id': 'dir-001',
 'paper_id': '088e1f601b1e',
 'run': 'cost_aware_agent_rl_20261011T034213Z',
 'source_pointer': '/result/candidates/0/papers/2/evidence_reviews/0/format_repairs/0'}


In [4]:
raw = example["invalid_output"]
parsed = _parse_json(raw) if isinstance(raw, str) else raw
print("JSON parsed. Top-level keys:", list(parsed))
schema = EvidenceReport.model_json_schema()
print("Allowed scope aspects:")
pprint(schema["$defs"]["ScopeCheck"]["properties"]["aspect"])

JSON parsed. Top-level keys: ['claim_checks', 'coverage_checks', 'summary', 'type']
Allowed scope aspects:
{'enum': ['result',
          'outcome',
          'model',
          'comparator',
          'dataset',
          'regime',
          'measurement',
          'assumption',
          'other'],
 'title': 'Aspect',
 'type': 'string'}


## Run the same Pydantic schema check
The error location identifies the exact nested field. `literal_error` means an unsupported enum value; `extra_forbidden` means an unexpected field. Inspect `schema` for the complete contract.

In [5]:
error = expect_failure(lambda: EvidenceReport.model_validate(parsed))
assert hasattr(error, "errors")
for issue in error.errors():
    pprint({key: issue[key] for key in ("loc", "type", "input", "msg")})

Expected ValidationError:
1 validation error for EvidenceReport
type
  Extra inputs are not permitted [type=extra_forbidden, input_value='json_object', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
{'input': 'json_object',
 'loc': ('type',),
 'msg': 'Extra inputs are not permitted',
 'type': 'extra_forbidden'}


No scientific conclusion follows from this failure. The response cannot yet be consumed under the declared schema. Changing an enum blindly can change meaning; a valid replacement needs to preserve the intended assertion.

## Try the same case through the Responses API
The saved failure above is a schema violation: an extra top-level `type` field. Below, reconstruct this example's input from saved run records and hash-checked local pages, then send the schema through `text.format` instead of embedding it in the user input.

The original wire request was not logged. Reconstruction uses the current production prompt/schema. These cells have not been executed. The live call defaults to off; set `RUN_MODEL_CALL = True` when you want to run it.

In [6]:
from notebooks.build_json_request import main as reconstruct_request

RUN_DIRECTORY = root.parent / "research_runs" / example["provenance"]["run"]
request = reconstruct_request(
    RUN_DIRECTORY,
    example_name="02_schema_violations",
    output_name="02_model_request",
)
system_prompt = request["system_prompt"]
user_input = request["user_input"]
user_input_new = user_input.copy()
del user_input_new["schema"]
print("Model:", request["model"])
print("Target IDs:", [t["target_id"] for t in user_input_new["payload"]["targets"]])
print("Historical error:", example["recorded_error"])

Saved reconstructed request; all source hashes and prior passages verified.
Model: deepseek-flash
Target IDs: ['dir-001-h04']
Historical error: 1 validation error for EvidenceReport
type
  Extra inputs are not permitted [type=extra_forbidden, input_value='json_object', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


In [7]:
user_input_new.keys()

dict_keys(['task', 'payload', 'record', 'required_scope_claim_ids'])

In [9]:
# user_input["schema"]

## Inspect the prompt and complete input
The schema is supplied separately to the API in the next cell.

In [ ]:
print(system_prompt)
print(json.dumps(user_input_new, indent=2, ensure_ascii=False))
print("Response schema:")
print(json.dumps(user_input["schema"], indent=2))

## Make the call yourself
This matches your notebook 01 call: `client.respond`, `instructions`, and `text.format.type = json_schema`, without adding `strict`. No automatic repair or SDK retries. The return value is a string, so it is saved directly. Current provider configuration and output limits apply.

In [11]:
RUN_MODEL_CALL = True
MODEL = request["model"]
MAX_OUTPUT_TOKENS = request["max_tokens"]
fresh_result = None
finish_reason = None
if RUN_MODEL_CALL:
    from llm_client import AsyncLLMClient
    from datetime import datetime, timezone
    call_input = json.dumps(user_input_new)
    output_format = {
        "format": {
            "type": "json_schema",
            "name": "expected_output",
            "schema": user_input["schema"],
        }
    }
    client = AsyncLLMClient(request["provider"], concurrency=1, max_retries=0)
    try:
        fresh_result = await client.respond(
            model=MODEL,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            instructions=system_prompt,
            input=call_input,
            text=output_format,
        )
        finish_reason = client.last_finish_reason
    finally:
        await client._client.close()
    output_dir = root / "notebooks" / "local_results"
    output_dir.mkdir(exist_ok=True)
    output_path = output_dir / ("02_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")
    output_path.write_text(json.dumps({
        "request": {
            "provider": request["provider"], "model": MODEL,
            "max_output_tokens": MAX_OUTPUT_TOKENS,
            "instructions": system_prompt, "input": call_input, "text": output_format,
        },
        "response_text": fresh_result,
        "finish_reason": finish_reason,
    }, indent=2, ensure_ascii=False))
    print("Saved:", output_path)
    print("Finish reason:", finish_reason)
else:
    print("Not called. Set RUN_MODEL_CALL = True when ready.")

Saved: /home/hema/research_finder/notebooks/local_results/02_20261011T170127863567Z.json
Finish reason: completed


## Check whether the schema error persists
Parsing and schema validation are reported separately from reference validation. Passing the schema does not imply the evidence judgments are correct.

In [12]:
if fresh_result is None:
    print("No fresh response yet.")
else:
    step = "output completeness"
    try:
        if finish_reason not in ("completed", "stop"):
            raise ValueError(f"Response completion not confirmed: {finish_reason!r}")
        step = "JSON parsing"
        parsed_fresh = _parse_json(fresh_result)
        print("JSON parsing: PASS")
        step = "schema validation"
        report_fresh = EvidenceReport.model_validate(parsed_fresh)
        print("Schema validation: PASS — the historical schema error did not recur in this response.")
        step = "evidence-reference validation"
        saved_record = EvidenceRecord.model_validate(user_input["record"])
        validate_evidence_report(
            report_fresh, saved_record,
            [Passage.model_validate(p) for p in user_input["payload"]["prior_passages"]],
            require_scope=True,
        )
        print("Evidence-reference validation: PASS")
        print("Scope citation gaps:", scope_citation_gaps(report_fresh, saved_record))
    except ValueError as error:
        print(step + ": FAIL")
        print(error)

JSON parsing: PASS
Schema validation: PASS — the historical schema error did not recur in this response.
Evidence-reference validation: PASS
Scope citation gaps: {'c1': ['088e1f601b1e#p3:02976858a21f', '088e1f601b1e#p3:da11861f7cfe'], 'c6': ['088e1f601b1e#p6:1305a3abc3ac'], 'c7': ['088e1f601b1e#p6:759e738a2084', '088e1f601b1e#p7:94f4c70428e7'], 'c9': ['088e1f601b1e#p8:eb10bde42577']}


In [13]:
report_fresh

EvidenceReport(claim_checks=[ClaimCheck(claim_id='c1', decision='revise', explanation="The reward formalization matches exactly: the attached text states that effective paths must optimize guidance effectiveness (IoI, IoR) and path feasibility (CTR) and 'naturally motivates a reward defined as a weighted sum of these metrics', giving Eq. 1 R_path = α·IoI + β·IoR + γ·CTR. However, the claim's second clause — that a policy π_θ is learned to generate high-reward paths via policy-gradient reinforcement learning — is not stated in the two attached passages; it is established only in §2.1 (p3:da11861f7cfe, p3:02976858a21f), which are not attached to this claim. That is a citation repair rather than a substantive correction, but under the attached-evidence rule the claim cannot be marked supported as written.", scope_checks=[ScopeCheck(aspect='other', assertion='Path quality is formalized as R_path = α·IoI + β·IoR + γ·CTR, with IoI and IoR quantifying guidance effectiveness and CTR quantifyin